In [8]:
# SO CONFIGURANDO AMBIENTE

# Instalação do Pyomo e do Solver GLPK (Comando para Google Colab ou VS Code Terminal)
!pip install pyomo

import pyomo.environ as pyo
from pyomo.opt import SolverFactory

# Criamos o objeto do solver que será usado mais adiante
opt = SolverFactory('glpk', executable='/opt/homebrew/bin/glpsol')

print("Ambiente configurado com sucesso!")

Ambiente configurado com sucesso!


In [9]:

# 1. Entrada de Dados
origens = ['Uberlandia', 'Rio_Verde']
destinos = ['Santos', 'Paranagua']

oferta = {'Uberlandia': 300, 'Rio_Verde': 500}
demanda = {'Santos': 400, 'Paranagua': 400}

custos = {
    ('Uberlandia', 'Santos'): 10, ('Uberlandia', 'Paranagua'): 15,
    ('Rio_Verde', 'Santos'): 12, ('Rio_Verde', 'Paranagua'): 11
}

# 2. Criação do Modelo
model = pyo.ConcreteModel()

# 3. Variáveis de Decisão: x[i,j] quantidade enviada de i para j
model.x = pyo.Var(origens, destinos, within=pyo.NonNegativeReals)

# 4. Função Objetivo: Minimização dos custos
model.obj = pyo.Objective(expr=sum(model.x[i,j] * custos[i,j] for i in origens for j in destinos), sense=pyo.minimize)

# 5. Restrições de Oferta
# O sum() percorre todas os destinos 'j' para a origem fixo 'i'
# Retorna para o Pyomo a relação de desigualdade. O solver verá algo como: (xi,1 + xi,2+...+xi,n ≤ oferta[i]).

def rest_oferta(model, i):
    return sum(model.x[i,j] for j in destinos) == oferta[i]
model.rest_oferta = pyo.Constraint(origens, rule=rest_oferta)

# 6. Restrições de Demanda
# O sum() percorre todas as origens 'i' para o destino fixo 'j'
# Retorna para o Pyomo a relação de desigualdade. O solver verá algo como: (x1,j + x2,j+...+ xn,j ≥ demanda[j]).

def rest_demanda(model, j):
    return sum(model.x[i,j] for i in origens) == demanda[j]
model.rest_demanda = pyo.Constraint(destinos, rule=rest_demanda)

# 7. ATIVAÇÃO DA SENSIBILIDADE (IMPORTANTE)
# O sufixo 'dual' permite acessar os Preços Sombra e Custos Reduzidos
model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

# 8. Solução
opt = pyo.SolverFactory('glpk', executable='/opt/homebrew/bin/glpsol')
results = opt.solve(model)

print(f"\nStatus da Solução: {results.solver.status}")
print(f"Valor Ótimo: {pyo.value(model.obj):.2f}")

# 9 .Exibir resultados
for i in origens:
    for j in destinos:
        if pyo.value(model.x[i,j]) > 0:
            print(f"Enviar {pyo.value(model.x[i,j])} de {i} para {j}")

# 10. Gerando o relatório de sensibilidade completo em arquivo texto

# O sufixo 'ranges' cria o relatório de sensibilidade textual do GLPK

opt.solve(model, options={'ranges': 'relatorio_sensibilidade.txt'})

# Lendo os intervalos de estabilidade do arquivo gerado

with open('relatorio_sensibilidade.txt', 'r') as f:
    # Mostra as linhas que contêm os limites de estabilidade (Upper e Lower bounds)
    print(f.read())


Status da Solução: ok
Valor Ótimo: 8600.00
Enviar 300.0 de Uberlandia para Santos
Enviar 100.0 de Rio_Verde para Santos
Enviar 400.0 de Rio_Verde para Paranagua
GLPK 5.0  - SENSITIVITY ANALYSIS REPORT                                                                         Page   1

Problem:    
Objective:  x1 = 8600 (MINimum)

   No. Row name     St      Activity         Slack   Lower bound       Activity      Obj coef  Obj value at Limiting
                                          Marginal   Upper bound          range         range   break point variable
------ ------------ -- ------------- ------------- -------------  ------------- ------------- ------------- ------------
     1 c_e_x6_      NS     300.00000        .          300.00000      300.00000          -Inf    8600.00000 c_e_x9_
                                           9.00000     300.00000      300.00000          +Inf    8600.00000 c_e_x9_

     2 c_e_x7_      NS     500.00000        .          500.00000      500.00000   